In [3]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [4]:
load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")

if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")
    

Gemini API Key exists and begins AQ.Ab8RN


In [5]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {"role": "user", "content": "Hello!"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I help you today?


In [6]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.
        Help users understand their pre-provided medication schedule.
        Do not invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": "What can you help me with?"
    }
]

In [7]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

print(response.choices[0].message.content)

I am your AI Medication Assistant, designed to help you manage and understand your prescribed medication schedule.

**Here is how I can assist you:**

*   **Schedule Clarification:** I can help you organize your dosages and timing so you know exactly when to take your medications based on the information you provide.
*   **Adherence Reminders:** I can help you understand how to structure your daily routine to stay consistent with your prescription instructions.
*   **Organization:** I can assist in categorizing your medications by time of day (e.g., morning, with food, bedtime) to make your regimen easier to follow.
*   **Terminology:** I can explain common medical abbreviations or terminology found on your prescription labels.

---

### **Important Limitations (Please read):**
*   **No Medical Advice:** I am not a doctor or a pharmacist. I cannot provide medical diagnoses or medical advice.
*   **No Dosage Changes:** I cannot change, recommend, or adjust your prescribed dosages. Alway

In [8]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {
            "role": "system",
            "content": """
            You are an AI Medication Assistant.

            Help users understand their pre-provided medication schedule.
            Only use the medication information provided to you.
            Never invent medication information or change a prescribed dosage.
            If the required information is not provided, say that you don't have that information.
            """
        },
        {
            "role": "user",
            "content": f"""
            Here is the user's medication information:

            {medication_context}

            What medicines does the user take in the morning?
            """
        }
    ]
)

print(response.choices[0].message.content)

NameError: name 'medication_context' is not defined

In [9]:
def get_medicines():
    result = (
        supabase
        .table("medications")
        .select("id, name, dosage, instructions, start_date, end_date")
        .execute()
    )

    return result.data
print(get_medicines())

NameError: name 'supabase' is not defined

In [ ]:
get_medicines_tool = {
    "type": "function",
    "function": {
        "name": "get_medicines",
        "description": "Get the user's complete medication list and schedule.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
}

In [ ]:
tools = [get_medicines_tool]

In [ ]:
print(tools)

[{'type': 'function', 'function': {'name': 'get_medicines', 'description': "Get the user's complete medication list and schedule.", 'parameters': {'type': 'object', 'properties': {}, 'required': []}}}]


In [ ]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {
            "role": "system",
            "content": """
            You are an AI Medication Assistant.
            When the user asks about their medications,
            use the available tool to retrieve their medication information.
            """
        },
        {
            "role": "user",
            "content": "What medicines am I currently taking?"
        }
    ],
    tools=tools
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_2716644', function=Function(arguments='{}', name='get_medicines'), type='function', extra_content={'google': {'thought_signature': 'EnEKbwERTTIPz5VRZmbb4FaX0OkTK8e0/drjjoALlC13gxF5sXRpzv3ptZozW5p96Viz+u/lyxpznnS8Zz6aI6rLt25UWhUsfNrjXmBvo1BO6Hfeq5UCIMucTmW+3Baj2mosmvfjkksI/v8VaUOrUetOMw=='}})])


In [ ]:
tool_call = response.choices[0].message.tool_calls[0]

print("Tool name:", tool_call.function.name)
print("Arguments:", tool_call.function.arguments)

Tool name: get_medicines
Arguments: {}


In [ ]:
tool_result = get_medicines()

print(tool_result)

[{'name': 'Paracetamol', 'dosage': '500 mg', 'time': '8:00 AM', 'instructions': 'Take after breakfast'}, {'name': 'Metformin', 'dosage': '500 mg', 'time': '8:00 PM', 'instructions': 'Take after dinner'}]


In [ ]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.
        Help users understand their pre-provided medication schedule.
        Only use the medication information provided by the tool.
        Never invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": "What medicines am I currently taking?"
    },
    response.choices[0].message,
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(tool_result)
    }
]

In [ ]:
final_response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages,
    tools=tools
)

print(final_response.choices[0].message.content)

You are currently taking the following medications:

*   **Paracetamol (500 mg):** Take at 8:00 AM after breakfast.
*   **Metformin (500 mg):** Take at 8:00 PM after dinner.


In [ ]:
available_tools = {
    "get_medicines": get_medicines
}

In [ ]:
tool_call = response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name
function_to_call = available_tools[function_name]

tool_result = function_to_call()

print(tool_result)

[{'name': 'Paracetamol', 'dosage': '500 mg', 'time': '8:00 AM', 'instructions': 'Take after breakfast'}, {'name': 'Metformin', 'dosage': '500 mg', 'time': '8:00 PM', 'instructions': 'Take after dinner'}]


In [ ]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {
            "role": "system",
            "content": """
            You are an AI Medication Assistant.
            When the user asks about their medications,
            use the available tool to retrieve their medication information.
            """
        },
        {
            "role": "user",
            "content": "What medicines am I currently taking?"
        }
    ],
    tools=tools
)

In [10]:
tool_call = response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name
function_to_call = available_tools[function_name]

tool_result = function_to_call()

print(tool_result)

TypeError: 'NoneType' object is not subscriptable

In [11]:
def get_medicine_by_name(name):
    result = (
        supabase
        .table("medications")
        .select("id, name, dosage, instructions, start_date, end_date")
        .ilike("name", name)
        .execute()
    )

    if not result.data:
        return {"error": "Medicine not found"}

    return result.data[0]

In [12]:
print(get_medicine_by_name("Metformin"))

NameError: name 'supabase' is not defined

In [ ]:
get_medicine_by_name_tool = {
    "type": "function",
    "function": {
        "name": "get_medicine_by_name",
        "description": "Get the details of a medicine by its name.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": "The name of the medicine."
                }
            },
            "required": ["name"]
        }
    }
}

In [ ]:
tools = [
    get_medicines_tool,
    get_medicine_by_name_tool
]

In [ ]:
available_tools = {
    "get_medicines": get_medicines,
    "get_medicine_by_name": get_medicine_by_name
}

In [ ]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {
            "role": "system",
            "content": """
            You are an AI Medication Assistant.
            When the user asks about a specific medicine,
            use the appropriate medication tool.
            """
        },
        {
            "role": "user",
            "content": "Tell me the details of Metformin."
        }
    ],
    tools=tools
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_242913', function=Function(arguments='{"name":"Metformin"}', name='get_medicine_by_name'), type='function', extra_content={'google': {'thought_signature': 'EnEKbwERTTIPDbTORSMBBUSoQ4T/uToOBqARCCO7yWO6/ggoC/vpscWhS3xS6KpPLsRmUTEYgTBEf3yReVF3AsEBVyMcyOeNAWpTRpfcIlog2r/nb7KpMTIkuuOOYyTwmzbbFlRdXSUR4YIlvvje03rrtQ=='}})])


In [ ]:
tool_call = response.choices[0].message.tool_calls[0]

print("Function:", tool_call.function.name)
print("Arguments:", tool_call.function.arguments)

Function: get_medicine_by_name
Arguments: {"name":"Metformin"}


In [ ]:
import json

arguments = json.loads(tool_call.function.arguments)

print(arguments)

{'name': 'Metformin'}


In [ ]:
tool_result = function_to_call(**arguments)

print(tool_result)

TypeError: get_medicines() got an unexpected keyword argument 'name'

In [ ]:
tool_call = response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

function_to_call = available_tools[function_name]

print("Function name:", function_name)
print("Arguments:", arguments)
print("Function to call:", function_to_call.__name__)

Function name: get_medicine_by_name
Arguments: {'name': 'Metformin'}
Function to call: get_medicine_by_name


In [ ]:
tool_result = function_to_call(**arguments)

print(tool_result)

{'name': 'Metformin', 'dosage': '500 mg', 'time': '8:00 PM', 'instructions': 'Take after dinner'}


In [ ]:
tool_call = response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

function_to_call = available_tools[function_name]

tool_result = function_to_call(**arguments)

In [ ]:
from supabase import create_client
import os
from dotenv import load_dotenv

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
result = supabase.table("medications").select("*").execute()

print(result.data)

[{'id': 7, 'user_id': '5839fe1f-14f0-44ee-b72f-d805dbdab7bf', 'name': 'Paracetamol', 'dosage': '500 mg', 'instructions': 'Take after breakfast', 'start_date': '2026-09-01', 'end_date': '2026-09-10', 'created_at': '2026-09-06T08:22:39.319331+00:00'}, {'id': 8, 'user_id': '5839fe1f-14f0-44ee-b72f-d805dbdab7bf', 'name': 'Metformin', 'dosage': '500 mg', 'instructions': 'Take after dinner', 'start_date': '2026-09-01', 'end_date': '2026-09-30', 'created_at': '2026-09-06T08:22:39.319331+00:00'}, {'id': 9, 'user_id': '5839fe1f-14f0-44ee-b72f-d805dbdab7bf', 'name': 'Vitamin D3', 'dosage': '1000 IU', 'instructions': 'Take with lunch', 'start_date': '2026-09-01', 'end_date': '2026-09-30', 'created_at': '2026-09-06T08:22:39.319331+00:00'}]


In [ ]:
result = supabase.table("medications").select("*").execute()

print(result.data)

[{'id': 7, 'user_id': '5839fe1f-14f0-44ee-b72f-d805dbdab7bf', 'name': 'Paracetamol', 'dosage': '500 mg', 'instructions': 'Take after breakfast', 'start_date': '2026-09-01', 'end_date': '2026-09-10', 'created_at': '2026-09-06T08:22:39.319331+00:00'}, {'id': 8, 'user_id': '5839fe1f-14f0-44ee-b72f-d805dbdab7bf', 'name': 'Metformin', 'dosage': '500 mg', 'instructions': 'Take after dinner', 'start_date': '2026-09-01', 'end_date': '2026-09-30', 'created_at': '2026-09-06T08:22:39.319331+00:00'}, {'id': 9, 'user_id': '5839fe1f-14f0-44ee-b72f-d805dbdab7bf', 'name': 'Vitamin D3', 'dosage': '1000 IU', 'instructions': 'Take with lunch', 'start_date': '2026-09-01', 'end_date': '2026-09-30', 'created_at': '2026-09-06T08:22:39.319331+00:00'}]


In [ ]:
result = (
    supabase
    .table("medications")
    .select("id, name, dosage, instructions, start_date, end_date")
    .execute()
)

print(result.data)

[{'id': 7, 'name': 'Paracetamol', 'dosage': '500 mg', 'instructions': 'Take after breakfast', 'start_date': '2026-09-01', 'end_date': '2026-09-10'}, {'id': 8, 'name': 'Metformin', 'dosage': '500 mg', 'instructions': 'Take after dinner', 'start_date': '2026-09-01', 'end_date': '2026-09-30'}, {'id': 9, 'name': 'Vitamin D3', 'dosage': '1000 IU', 'instructions': 'Take with lunch', 'start_date': '2026-09-01', 'end_date': '2026-09-30'}]


In [ ]:
result = (
    supabase
    .table("medication_schedules")
    .select("medication_id, scheduled_time, frequency")
    .execute()
)

print(result.data)

[{'medication_id': 7, 'scheduled_time': '08:00:00', 'frequency': 'daily'}, {'medication_id': 9, 'scheduled_time': '13:00:00', 'frequency': 'daily'}, {'medication_id': 8, 'scheduled_time': '20:00:00', 'frequency': 'daily'}]


In [ ]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.

        Use the available tools whenever medication information is required.
        Only use information returned by the tools.
        Never invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": """
        Which tablet shld i take mrng
    
        """
    }
]

while True:

    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    if not message.tool_calls:
        print(message.content)
        break

    messages.append(message)

    for tool_call in message.tool_calls:

        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        function_to_call = available_tools[function_name]

        tool_result = function_to_call(**arguments)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result)
        })

According to your medication schedule, you should take **Paracetamol (500 mg)** in the morning at **8:00 AM**. Please remember to take it after breakfast.


In [ ]:
from datetime import datetime

def get_current_time():
    return datetime.now().strftime("%H:%M")

In [ ]:
get_current_time()

'15:52'

In [ ]:
def get_due_medicines(time):
    result = (
        supabase
        .table("medication_schedules")
        .select(
            "scheduled_time, frequency, medications(id, name, dosage, instructions, start_date, end_date)"
        )
        .eq("scheduled_time", time)
        .execute()
    )

    return result.data

In [ ]:
get_due_medicines("08:00:00")

[{'scheduled_time': '08:00:00',
  'frequency': 'daily',
  'medications': {'id': 7,
   'name': 'Paracetamol',
   'dosage': '500 mg',
   'end_date': '2026-09-10',
   'start_date': '2026-09-01',
   'instructions': 'Take after breakfast'}}]

In [ ]:
def get_due_medicines():
    current_time = datetime.now().strftime("%H:00:00")

    result = (
        supabase
        .table("medication_schedules")
        .select(
            "scheduled_time, frequency, medications(id, name, dosage, instructions, start_date, end_date)"
        )
        .eq("scheduled_time", current_time)
        .execute()
    )

    return result.data

In [ ]:
get_due_medicines()

[]

In [ ]:
get_due_medicines_tool = {
    "type": "function",
    "function": {
        "name": "get_due_medicines",
        "description": "Get the medicines scheduled for the current time.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
}

In [ ]:
tools = [
    get_medicines_tool,
    get_medicine_by_name_tool,
    get_due_medicines_tool
]

In [ ]:
available_tools = {
    "get_medicines": get_medicines,
    "get_medicine_by_name": get_medicine_by_name,
    "get_due_medicines": get_due_medicines
}

In [ ]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.
        Use the available tools whenever medication information is required.
        Only use information returned by the tools.
        Never invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": "What medicine should I take now?"
    }
]

In [ ]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.

        Use the available tools whenever medication information is required.
        Only use information returned by the tools.
        Never invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": """
        "Till how long shld i take paracetamol"
        """
    }
]

while True:

    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    if not message.tool_calls:
        print(message.content)
        break

    messages.append(message)

    for tool_call in message.tool_calls:

        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        function_to_call = available_tools[function_name]

        tool_result = function_to_call(**arguments)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result)
        })

According to your records, you are prescribed Paracetamol (500 mg) to be taken from September 1, 2026, until September 10, 2026. 

Please follow the dates specified in your prescription. If you have any concerns about the duration of your treatment, it is best to consult your doctor.


In [ ]:
def get_medication_information(name):
    result = (
        supabase
        .table("medications")
        .select(
            "name, dosage, purpose, prescribed_for, instructions, start_date, end_date"
        )
        .ilike("name", name)
        .execute()
    )

    if not result.data:
        return {"error": "Medicine not found"}

    return result.data[0]

In [ ]:
get_medication_information("Metformin")

{'name': 'Metformin',
 'dosage': '500 mg',
 'purpose': 'Blood sugar management',
 'prescribed_for': 'Blood sugar control',
 'instructions': 'Take after dinner',
 'start_date': '2026-09-01',
 'end_date': '2026-09-30'}

In [ ]:
get_medication_information_tool = {
    "type": "function",
    "function": {
        "name": "get_medication_information",
        "description": "Get detailed information about a specific medicine, including its purpose and what it was prescribed for.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": "The name of the medicine."
                }
            },
            "required": ["name"]
        }
    }
}

In [ ]:
tools = [
    get_medicines_tool,
    get_medicine_by_name_tool,
    get_due_medicines_tool,
    get_medication_information_tool
]

In [ ]:
available_tools = {
    "get_medicines": get_medicines,
    "get_medicine_by_name": get_medicine_by_name,
    "get_due_medicines": get_due_medicines,
    "get_medication_information": get_medication_information
}

In [ ]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.

        Use the available tools whenever medication information is required.
        Only use information returned by the tools.
        Never invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": """
        “What medicine should I take now, and why was it prescribed?”
        """
    }
]

while True:

    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    if not message.tool_calls:
        print(message.content)
        break

    messages.append(message)

    for tool_call in message.tool_calls:

        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        function_to_call = available_tools[function_name]

        tool_result = function_to_call(**arguments)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result)
        })

You do not have any medicines due at this time. 

Here is your current medication list, their purposes, and when they are scheduled:

*   **Paracetamol (500 mg):** Prescribed for pain relief. Take this after breakfast.
*   **Vitamin D3 (1000 IU):** Prescribed for Vitamin D deficiency. Take this with lunch.
*   **Metformin (500 mg):** Prescribed for blood sugar control. Take this after dinner.


In [ ]:
from datetime import datetime

def get_due_medicines():
    current_time = datetime.now().strftime("%H:00:00")
    current_date = datetime.now().date().isoformat()

    result = (
        supabase
        .table("medication_schedules")
        .select(
            "scheduled_time, frequency, medications(id, name, dosage, instructions, start_date, end_date)"
        )
        .eq("scheduled_time", current_time)
        .execute()
    )

    due_medicines = []

    for item in result.data:
        medicine = item["medications"]

        if (
            medicine["start_date"] <= current_date
            and (
                medicine["end_date"] is None
                or current_date <= medicine["end_date"]
            )
        ):
            due_medicines.append(item)

    return due_medicines

In [ ]:
get_due_medicines()

[]

In [ ]:
from datetime import date

today = date.today().isoformat()
print(today)

2026-09-06


In [ ]:
def get_medicine_by_name(name):
    result = (
        supabase
        .table("medications")
        .select(
            "id, name, dosage, instructions, start_date, end_date, "
            "medication_schedules(scheduled_time, frequency)"
        )
        .ilike("name", name)
        .execute()
    )

    if not result.data:
        return {"error": "Medicine not found"}

    return result.data[0]

In [ ]:
get_medicine_by_name("Metformin")

{'id': 8,
 'name': 'Metformin',
 'dosage': '500 mg',
 'instructions': 'Take after dinner',
 'start_date': '2026-09-01',
 'end_date': '2026-09-30',
 'scheduled_time': '20:00:00',
 'frequency': 'daily'}

In [ ]:
def get_medicine_by_name(name):
    result = (
        supabase
        .table("medications")
        .select(
            "id, name, dosage, instructions, start_date, end_date, "
            "medication_schedules(scheduled_time, frequency)"
        )
        .ilike("name", name)
        .execute()
    )

    if not result.data:
        return {"error": "Medicine not found"}

    medicine = result.data[0]
    schedule = medicine.pop("medication_schedules", [])

    if schedule:
        medicine["scheduled_time"] = schedule[0]["scheduled_time"]
        medicine["frequency"] = schedule[0]["frequency"]

    return medicine

In [ ]:
get_medicine_by_name("Metformin")

{'id': 8,
 'name': 'Metformin',
 'dosage': '500 mg',
 'instructions': 'Take after dinner',
 'start_date': '2026-09-01',
 'end_date': '2026-09-30',
 'scheduled_time': '20:00:00',
 'frequency': 'daily'}

In [ ]:
def get_medication_information(name):

    result = (
        supabase
        .table("medications")
        .select(
            "name, dosage, purpose, prescribed_for, instructions, start_date, end_date, "
            "medication_schedules(scheduled_time, frequency)"
        )
        .ilike("name", name)
        .execute()
    )

    if not result.data:
        return {"error": "Medicine not found"}

    medicine = result.data[0]

    schedule = medicine.pop("medication_schedules", [])

    if schedule:
        medicine["scheduled_time"] = schedule[0]["scheduled_time"]
        medicine["frequency"] = schedule[0]["frequency"]

    return medicine

In [ ]:
get_medication_information("Metformin")

{'name': 'Metformin',
 'dosage': '500 mg',
 'purpose': 'Blood sugar management',
 'prescribed_for': 'Blood sugar control',
 'instructions': 'Take after dinner',
 'start_date': '2026-09-01',
 'end_date': '2026-09-30',
 'scheduled_time': '20:00:00',
 'frequency': 'daily'}

In [14]:
def get_medicines():
    result = (
        supabase
        .table("medications")
        .select(
            """
            id,
            name,
            dosage,
            purpose,
            prescribed_for,
            instructions,
            medication_schedules(
                id,
                dose_quantity,
                dose_unit,
                scheduled_time,
                frequency_type,
                frequency_count,
                frequency_unit,
                duration_value,
                duration_unit,
                day_of_week,
                meal_relation,
                meal_offset_minutes,
                start_date,
                end_date
            )
            """
        )
        .execute()
    )

    return result.data

In [15]:
medicines = get_medicines()

medicines

[{'id': 1,
  'name': 'Paracetamol',
  'dosage': '500 mg',
  'purpose': 'Pain relief',
  'prescribed_for': 'Pain',
  'instructions': 'Take after breakfast',
  'medication_schedules': [{'id': 1,
    'end_date': '2026-09-10',
    'dose_unit': 'tablet',
    'start_date': '2026-09-01',
    'day_of_week': None,
    'dose_quantity': 1,
    'duration_unit': 'day',
    'meal_relation': 'after',
    'duration_value': 10,
    'frequency_type': 'times_per',
    'frequency_unit': 'day',
    'scheduled_time': '08:00:00',
    'frequency_count': 1,
    'meal_offset_minutes': None}]},
 {'id': 2,
  'name': 'Metformin',
  'dosage': '500 mg',
  'purpose': 'Blood sugar management',
  'prescribed_for': 'Blood sugar control',
  'instructions': 'Take after meals',
  'medication_schedules': [{'id': 2,
    'end_date': '2026-09-30',
    'dose_unit': 'tablet',
    'start_date': '2026-09-01',
    'day_of_week': None,
    'dose_quantity': 1,
    'duration_unit': 'day',
    'meal_relation': 'after',
    'duration_va

In [13]:
import os
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

In [16]:
def get_medicines():
    result = (
        supabase
        .table("medications")
        .select(
            """
            id,
            name,
            dosage,
            purpose,
            prescribed_for,
            instructions,
            medication_schedules(
                id,
                dose_quantity,
                dose_unit,
                scheduled_time,
                frequency_type,
                frequency_count,
                frequency_unit,
                duration_value,
                duration_unit,
                day_of_week,
                meal_relation,
                meal_offset_minutes,
                start_date,
                end_date
            )
            """
        )
        .execute()
    )

    medicines = []

    for medicine in result.data:
        schedules = medicine.pop("medication_schedules", [])

        medicine["schedules"] = schedules

        medicines.append(medicine)

    return medicines

In [17]:
medicines = get_medicines()

medicines

[{'id': 1,
  'name': 'Paracetamol',
  'dosage': '500 mg',
  'purpose': 'Pain relief',
  'prescribed_for': 'Pain',
  'instructions': 'Take after breakfast',
  'schedules': [{'id': 1,
    'end_date': '2026-09-10',
    'dose_unit': 'tablet',
    'start_date': '2026-09-01',
    'day_of_week': None,
    'dose_quantity': 1,
    'duration_unit': 'day',
    'meal_relation': 'after',
    'duration_value': 10,
    'frequency_type': 'times_per',
    'frequency_unit': 'day',
    'scheduled_time': '08:00:00',
    'frequency_count': 1,
    'meal_offset_minutes': None}]},
 {'id': 2,
  'name': 'Metformin',
  'dosage': '500 mg',
  'purpose': 'Blood sugar management',
  'prescribed_for': 'Blood sugar control',
  'instructions': 'Take after meals',
  'schedules': [{'id': 2,
    'end_date': '2026-09-30',
    'dose_unit': 'tablet',
    'start_date': '2026-09-01',
    'day_of_week': None,
    'dose_quantity': 1,
    'duration_unit': 'day',
    'meal_relation': 'after',
    'duration_value': 30,
    'frequen

In [18]:
def get_medicine_by_name(name):
    result = (
        supabase
        .table("medications")
        .select(
            """
            id,
            name,
            dosage,
            purpose,
            prescribed_for,
            instructions,
            medication_schedules(
                id,
                dose_quantity,
                dose_unit,
                scheduled_time,
                frequency_type,
                frequency_count,
                frequency_unit,
                duration_value,
                duration_unit,
                day_of_week,
                meal_relation,
                meal_offset_minutes,
                start_date,
                end_date
            )
            """
        )
        .ilike("name", name)
        .execute()
    )

    if not result.data:
        return {"error": "Medicine not found"}

    medicine = result.data[0]

    schedules = medicine.pop("medication_schedules", [])

    medicine["schedules"] = schedules

    return medicine

In [19]:
get_medicine_by_name("Metformin")

{'id': 2,
 'name': 'Metformin',
 'dosage': '500 mg',
 'purpose': 'Blood sugar management',
 'prescribed_for': 'Blood sugar control',
 'instructions': 'Take after meals',
 'schedules': [{'id': 2,
   'end_date': '2026-09-30',
   'dose_unit': 'tablet',
   'start_date': '2026-09-01',
   'day_of_week': None,
   'dose_quantity': 1,
   'duration_unit': 'day',
   'meal_relation': 'after',
   'duration_value': 30,
   'frequency_type': 'times_per',
   'frequency_unit': 'day',
   'scheduled_time': '08:00:00',
   'frequency_count': 2,
   'meal_offset_minutes': None},
  {'id': 3,
   'end_date': '2026-09-30',
   'dose_unit': 'tablet',
   'start_date': '2026-09-01',
   'day_of_week': None,
   'dose_quantity': 1,
   'duration_unit': 'day',
   'meal_relation': 'after',
   'duration_value': 30,
   'frequency_type': 'times_per',
   'frequency_unit': 'day',
   'scheduled_time': '20:00:00',
   'frequency_count': 2,
   'meal_offset_minutes': None}]}

In [20]:
from datetime import datetime

now = datetime.now()

current_date = now.date()
current_time = now.strftime("%H:%M")
current_day = now.strftime("%A")

print("Current date:", current_date)
print("Current time:", current_time)
print("Current day:", current_day)

Current date: 2026-09-08
Current time: 17:37
Current day: Tuesday


In [21]:
from datetime import datetime


def get_due_medicines():

    now = datetime.now()

    current_date = now.date()
    current_time = now.strftime("%H:00:00")
    current_day = now.strftime("%A")

    result = (
        supabase
        .table("medication_schedules")
        .select(
            """
            id,
            medication_id,
            dose_quantity,
            dose_unit,
            scheduled_time,
            frequency_type,
            frequency_count,
            frequency_unit,
            duration_value,
            duration_unit,
            day_of_week,
            meal_relation,
            meal_offset_minutes,
            start_date,
            end_date,
            medications(
                id,
                name,
                dosage,
                purpose,
                prescribed_for,
                instructions
            )
            """
        )
        .eq("scheduled_time", current_time)
        .execute()
    )

    due_medicines = []

    for item in result.data:

        # Check whether the medicine is active today
        start_date = item["start_date"]
        end_date = item["end_date"]

        if start_date and current_date.isoformat() < start_date:
            continue

        if end_date and current_date.isoformat() > end_date:
            continue

        # Check specific weekday if one is provided
        day_of_week = item["day_of_week"]

        if day_of_week and day_of_week != current_day:
            continue

        # Add the medicine information to the schedule
        medicine = item["medications"]

        due_medicines.append({
            "medicine_id": medicine["id"],
            "name": medicine["name"],
            "dosage": medicine["dosage"],
            "purpose": medicine["purpose"],
            "prescribed_for": medicine["prescribed_for"],
            "instructions": medicine["instructions"],
            "schedule": {
                "schedule_id": item["id"],
                "dose_quantity": item["dose_quantity"],
                "dose_unit": item["dose_unit"],
                "scheduled_time": item["scheduled_time"],
                "frequency_type": item["frequency_type"],
                "frequency_count": item["frequency_count"],
                "frequency_unit": item["frequency_unit"],
                "duration_value": item["duration_value"],
                "duration_unit": item["duration_unit"],
                "day_of_week": item["day_of_week"],
                "meal_relation": item["meal_relation"],
                "meal_offset_minutes": item["meal_offset_minutes"],
                "start_date": item["start_date"],
                "end_date": item["end_date"]
            }
        })

    return due_medicines

In [22]:
get_due_medicines()

[]

In [26]:
get_due_medicines_at("08:00:00")

[{'medicine_id': 1,
  'name': 'Paracetamol',
  'dosage': '500 mg',
  'purpose': 'Pain relief',
  'prescribed_for': 'Pain',
  'instructions': 'Take after breakfast',
  'schedule': {'schedule_id': 1,
   'dose_quantity': 1,
   'dose_unit': 'tablet',
   'scheduled_time': '08:00:00',
   'frequency_type': 'times_per',
   'frequency_count': 1,
   'frequency_unit': 'day',
   'duration_value': 10,
   'duration_unit': 'day',
   'day_of_week': None,
   'meal_relation': 'after',
   'meal_offset_minutes': None,
   'start_date': '2026-09-01',
   'end_date': '2026-09-10'}},
 {'medicine_id': 2,
  'name': 'Metformin',
  'dosage': '500 mg',
  'purpose': 'Blood sugar management',
  'prescribed_for': 'Blood sugar control',
  'instructions': 'Take after meals',
  'schedule': {'schedule_id': 2,
   'dose_quantity': 1,
   'dose_unit': 'tablet',
   'scheduled_time': '08:00:00',
   'frequency_type': 'times_per',
   'frequency_count': 2,
   'frequency_unit': 'day',
   'duration_value': 30,
   'duration_unit': 'd

In [28]:
def get_medicines_by_day(day):

    result = (
        supabase
        .table("medication_schedules")
        .select(
            """
            id,
            medication_id,
            dose_quantity,
            dose_unit,
            scheduled_time,
            frequency_type,
            frequency_count,
            frequency_unit,
            duration_value,
            duration_unit,
            day_of_week,
            meal_relation,
            meal_offset_minutes,
            start_date,
            end_date,
            medications(
                id,
                name,
                dosage,
                purpose,
                prescribed_for,
                instructions
            )
            """
        )
        .execute()
    )

    medicines = []

    for item in result.data:

        # Daily medicines
        if (
            item["frequency_type"] == "times_per"
            and item["frequency_unit"] == "day"
        ):
            pass

        # Medicines assigned to a specific day
        elif item["day_of_week"]:
            if item["day_of_week"].lower() != day.lower():
                continue

        # Weekly frequency with no specific day
        else:
            continue

        medicine = item["medications"]

        medicines.append({
            "medicine_id": medicine["id"],
            "name": medicine["name"],
            "dosage": medicine["dosage"],
            "purpose": medicine["purpose"],
            "prescribed_for": medicine["prescribed_for"],
            "instructions": medicine["instructions"],
            "schedule": {
                "schedule_id": item["id"],
                "dose_quantity": item["dose_quantity"],
                "dose_unit": item["dose_unit"],
                "scheduled_time": item["scheduled_time"],
                "frequency_type": item["frequency_type"],
                "frequency_count": item["frequency_count"],
                "frequency_unit": item["frequency_unit"],
                "duration_value": item["duration_value"],
                "duration_unit": item["duration_unit"],
                "day_of_week": item["day_of_week"],
                "meal_relation": item["meal_relation"],
                "meal_offset_minutes": item["meal_offset_minutes"],
                "start_date": item["start_date"],
                "end_date": item["end_date"]
            }
        })

    return medicines

In [29]:
get_medicines_by_day("Friday")

[{'medicine_id': 1,
  'name': 'Paracetamol',
  'dosage': '500 mg',
  'purpose': 'Pain relief',
  'prescribed_for': 'Pain',
  'instructions': 'Take after breakfast',
  'schedule': {'schedule_id': 1,
   'dose_quantity': 1,
   'dose_unit': 'tablet',
   'scheduled_time': '08:00:00',
   'frequency_type': 'times_per',
   'frequency_count': 1,
   'frequency_unit': 'day',
   'duration_value': 10,
   'duration_unit': 'day',
   'day_of_week': None,
   'meal_relation': 'after',
   'meal_offset_minutes': None,
   'start_date': '2026-09-01',
   'end_date': '2026-09-10'}},
 {'medicine_id': 2,
  'name': 'Metformin',
  'dosage': '500 mg',
  'purpose': 'Blood sugar management',
  'prescribed_for': 'Blood sugar control',
  'instructions': 'Take after meals',
  'schedule': {'schedule_id': 2,
   'dose_quantity': 1,
   'dose_unit': 'tablet',
   'scheduled_time': '08:00:00',
   'frequency_type': 'times_per',
   'frequency_count': 2,
   'frequency_unit': 'day',
   'duration_value': 30,
   'duration_unit': 'd

In [30]:
get_medicines_by_day("Friday")

[{'medicine_id': 1,
  'name': 'Paracetamol',
  'dosage': '500 mg',
  'purpose': 'Pain relief',
  'prescribed_for': 'Pain',
  'instructions': 'Take after breakfast',
  'schedule': {'schedule_id': 1,
   'dose_quantity': 1,
   'dose_unit': 'tablet',
   'scheduled_time': '08:00:00',
   'frequency_type': 'times_per',
   'frequency_count': 1,
   'frequency_unit': 'day',
   'duration_value': 10,
   'duration_unit': 'day',
   'day_of_week': None,
   'meal_relation': 'after',
   'meal_offset_minutes': None,
   'start_date': '2026-09-01',
   'end_date': '2026-09-10'}},
 {'medicine_id': 2,
  'name': 'Metformin',
  'dosage': '500 mg',
  'purpose': 'Blood sugar management',
  'prescribed_for': 'Blood sugar control',
  'instructions': 'Take after meals',
  'schedule': {'schedule_id': 2,
   'dose_quantity': 1,
   'dose_unit': 'tablet',
   'scheduled_time': '08:00:00',
   'frequency_type': 'times_per',
   'frequency_count': 2,
   'frequency_unit': 'day',
   'duration_value': 30,
   'duration_unit': 'd

In [31]:
def get_medicines_by_time(time):

    current_date = datetime.now().date().isoformat()
    current_day = datetime.now().strftime("%A")

    result = (
        supabase
        .table("medication_schedules")
        .select(
            """
            id,
            medication_id,
            dose_quantity,
            dose_unit,
            scheduled_time,
            frequency_type,
            frequency_count,
            frequency_unit,
            duration_value,
            duration_unit,
            day_of_week,
            meal_relation,
            meal_offset_minutes,
            start_date,
            end_date,
            medications(
                id,
                name,
                dosage,
                purpose,
                prescribed_for,
                instructions
            )
            """
        )
        .eq("scheduled_time", time)
        .execute()
    )

    medicines = []

    for item in result.data:

        # Check whether the schedule is active today
        start_date = item["start_date"]
        end_date = item["end_date"]

        if start_date and current_date < start_date:
            continue

        if end_date and current_date > end_date:
            continue

        # Check specific weekday
        if item["day_of_week"]:
            if item["day_of_week"].lower() != current_day.lower():
                continue

        # Weekly schedules without specific days
        elif (
            item["frequency_type"] == "times_per"
            and item["frequency_unit"] == "week"
        ):
            continue

        medicine = item["medications"]

        medicines.append({
            "medicine_id": medicine["id"],
            "name": medicine["name"],
            "dosage": medicine["dosage"],
            "purpose": medicine["purpose"],
            "prescribed_for": medicine["prescribed_for"],
            "instructions": medicine["instructions"],
            "schedule": {
                "schedule_id": item["id"],
                "dose_quantity": item["dose_quantity"],
                "dose_unit": item["dose_unit"],
                "scheduled_time": item["scheduled_time"],
                "frequency_type": item["frequency_type"],
                "frequency_count": item["frequency_count"],
                "frequency_unit": item["frequency_unit"],
                "duration_value": item["duration_value"],
                "duration_unit": item["duration_unit"],
                "day_of_week": item["day_of_week"],
                "meal_relation": item["meal_relation"],
                "meal_offset_minutes": item["meal_offset_minutes"],
                "start_date": item["start_date"],
                "end_date": item["end_date"]
            }
        })

    return medicines

In [32]:
get_medicines_by_time("20:00:00")

[{'medicine_id': 2,
  'name': 'Metformin',
  'dosage': '500 mg',
  'purpose': 'Blood sugar management',
  'prescribed_for': 'Blood sugar control',
  'instructions': 'Take after meals',
  'schedule': {'schedule_id': 3,
   'dose_quantity': 1,
   'dose_unit': 'tablet',
   'scheduled_time': '20:00:00',
   'frequency_type': 'times_per',
   'frequency_count': 2,
   'frequency_unit': 'day',
   'duration_value': 30,
   'duration_unit': 'day',
   'day_of_week': None,
   'meal_relation': 'after',
   'meal_offset_minutes': None,
   'start_date': '2026-09-01',
   'end_date': '2026-09-30'}},
 {'medicine_id': 3,
  'name': 'Amoxicillin',
  'dosage': '500 mg',
  'purpose': 'Antibacterial treatment',
  'prescribed_for': 'Bacterial infection',
  'instructions': 'Take after food',
  'schedule': {'schedule_id': 6,
   'dose_quantity': 1,
   'dose_unit': 'capsule',
   'scheduled_time': '20:00:00',
   'frequency_type': 'times_per',
   'frequency_count': 3,
   'frequency_unit': 'day',
   'duration_value': 10,